# Week 8 — Regularization, Hyperparameter Tuning & Optimization
## Notebook 1: Overfitting, Underfitting & the Bias-Variance Tradeoff

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated Time:** 2.5 hours &nbsp;&nbsp;|&nbsp;&nbsp; **Theme:** California-Style House Price Prediction

---

## Why This Matters

You can build a model that achieves **zero training error** and still be completely useless in production. The gap between *memorizing* training data and *learning* from it is the most important concept in all of applied ML. Every regularization technique, every cross-validation strategy, every dropout layer in a neural network — all of it exists to solve the problem you'll understand in this notebook.

---

## Real-World Analogy First

Imagine three real estate agents tasked with predicting house prices in a new neighborhood:

**Agent 1 — The Memorizer (Overfitting):**  
She spent months memorizing every single house sale from 2019 — the exact price of each house, down to the dollar. She can tell you the price of any 2019 house instantly and perfectly. But when you show her a 2024 house, she's lost — her knowledge is too specific to past data. She memorized noise along with signal.

**Agent 2 — The Simpleton (Underfitting):**  
He only knows one rule: "bigger house = more expensive." He ignores location, condition, school district, and year built. His predictions are systematically off — too high for rundown houses in bad areas, too low for small renovated homes in prime spots. His mental model is too simple to capture real market dynamics.

**Agent 3 — The Market Expert (Just Right):**  
She learned the *patterns* — how square footage, neighborhood, condition, and proximity to schools interact to determine price. She doesn't remember individual houses, but she understands the market. She generalizes well to new houses she's never seen.

> **Key insight:** The memorizer fails on 2024 houses because she fit *noise* in 2019 data. The simpleton always fails because her model is too constrained. The expert generalizes because she captured the true underlying signal.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Define overfitting and underfitting precisely and diagnose them from learning curves
2. Decompose prediction error into bias², variance, and irreducible noise
3. Use polynomial regression to demonstrate the bias-variance tradeoff empirically
4. Identify the "sweet spot" of model complexity on a validation curve
5. Explain why regularization is the primary cure for overfitting

## Section 1: Setup & Dataset Creation

We create a synthetic California-style house price dataset with a known true relationship plus noise. This lets us *know the ground truth* and see exactly what overfitting and underfitting look like.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Reproducibility and aesthetics
np.random.seed(42)
sns.set_theme(style='whitegrid')
COLORS = {
    'train': '#2196F3',    # blue
    'val': '#F44336',      # red
    'true': '#4CAF50',     # green
    'pred': '#FF9800',     # orange
    'highlight': '#9C27B0' # purple
}
print('Libraries loaded successfully.')
print(f'NumPy: {np.__version__}  |  Pandas: {pd.__version__}')

In [ ]:
# ---------------------------------------------------------------------------
# Dataset: California-style house prices
# True relationship: nonlinear function of sqft + small contributions from
# other features + Gaussian noise.
# ---------------------------------------------------------------------------
np.random.seed(42)
N = 150

# Primary feature we'll use for polynomial demos (normalized 0-1)
sqft_raw = np.random.uniform(600, 4000, N)          # square feet
sqft     = (sqft_raw - sqft_raw.min()) / (sqft_raw.max() - sqft_raw.min())  # 0 to 1

bedrooms   = np.random.randint(1, 6, N).astype(float)
bathrooms  = np.clip(np.random.normal(2.5, 0.8, N), 1, 5)
age        = np.random.randint(1, 60, N).astype(float)      # years old
garage     = np.random.randint(0, 4, N).astype(float)       # garage spaces

# True price function (nonlinear in sqft, additive other features)
# Price in $100k: cubic-ish curve
true_price = (
    3.0
    + 8.0  * sqft
    - 6.0  * sqft**2
    + 10.0 * sqft**3
    + 0.3  * bedrooms
    + 0.5  * bathrooms
    - 0.02 * age
    + 0.2  * garage
)

noise       = np.random.normal(0, 0.4, N)
price       = true_price + noise

df = pd.DataFrame({
    'sqft': sqft_raw,
    'sqft_norm': sqft,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age': age,
    'garage': garage,
    'price_100k': price
})

print(f'Dataset shape: {df.shape}')
print(f'Price range: ${price.min()*100:.0f}k — ${price.max()*100:.0f}k')
print()
df.describe().round(3)

## Section 2: Formal Definitions

### Overfitting
A model **overfits** when it is so complex that it captures not just the true signal but also the random noise in the training data.

- **Symptom:** Training error ≈ 0, validation/test error >> training error
- **Cause:** Model has too many parameters relative to the amount of data
- **Analogy:** The memorizer agent — perfect on 2019 houses, terrible on 2024 houses

### Underfitting
A model **underfits** when it is too simple to capture the true relationship between features and target.

- **Symptom:** High training error AND high validation error (both large, roughly equal)
- **Cause:** Model is too constrained (too few parameters, or wrong functional form)
- **Analogy:** The simpleton agent — wrong on everything because "bigger = more expensive" misses too much

### The Diagnostic Formula
```
If train_error ≈ val_error AND both high  →  UNDERFITTING (high bias)
If train_error << val_error               →  OVERFITTING  (high variance)
If train_error ≈ val_error AND both low   →  GOOD FIT
```

## Section 3: The Bias-Variance Decomposition

For any prediction model, the expected test error on a new point $x$ decomposes as:

$$\text{Expected Error} = \underbrace{\text{Bias}^2}_{\text{systematic error}} + \underbrace{\text{Variance}}_{\text{sensitivity to training data}} + \underbrace{\sigma^2}_{\text{irreducible noise}}$$

**Bias²** — How far off is the average prediction from the truth?  
- High bias = model is systematically wrong (underfitting)  
- The memorizer has low bias on training data but we measure bias on *new* data

**Variance** — How much does the prediction change if we retrain on a different dataset?  
- High variance = model is very sensitive to which specific training samples we happened to draw (overfitting)  
- The memorizer has extremely high variance: tiny changes in training data → wildly different model

**Irreducible noise σ²** — Randomness in the data that no model can remove  
- This is the floor: no matter how good your model, you can't go below σ²

### The Tradeoff
As model complexity increases:
- Bias² **decreases** (model can fit more complex functions)
- Variance **increases** (model becomes more sensitive to training data)
- Optimal complexity = minimum total error = the "sweet spot"

In [ ]:
# ---------------------------------------------------------------------------
# Section 3: Polynomial Regression at Multiple Degrees
# We use only sqft_norm → price to keep things 1D and visualizable.
# ---------------------------------------------------------------------------
X_1d = df['sqft_norm'].values.reshape(-1, 1)
y    = df['price_100k'].values

X_train, X_val, y_train, y_val = train_test_split(
    X_1d, y, test_size=0.25, random_state=42
)
print(f'Train size: {len(X_train)}  |  Validation size: {len(X_val)}')

# Degrees to evaluate
degrees = [1, 3, 5, 8, 12, 15]

# Store results
results = {'degree': [], 'train_mse': [], 'val_mse': []}
fitted_models = {}

for deg in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, pipe.predict(X_train))
    val_mse   = mean_squared_error(y_val,   pipe.predict(X_val))
    results['degree'].append(deg)
    results['train_mse'].append(train_mse)
    results['val_mse'].append(val_mse)
    fitted_models[deg] = pipe

df_results = pd.DataFrame(results)
print('\nMSE Results by Polynomial Degree:')
print(df_results.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ---------------------------------------------------------------------------
# Section 4: Complexity Curve — Train MSE and Val MSE vs Degree
# This is the classic bias-variance tradeoff visualization.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_results['degree'], df_results['train_mse'],
        marker='o', linewidth=2.5, color=COLORS['train'],
        label='Training MSE', markersize=8)
ax.plot(df_results['degree'], df_results['val_mse'],
        marker='s', linewidth=2.5, color=COLORS['val'],
        label='Validation MSE', markersize=8)

# Find sweet spot
sweet_idx = df_results['val_mse'].idxmin()
sweet_deg = df_results.loc[sweet_idx, 'degree']
sweet_mse = df_results.loc[sweet_idx, 'val_mse']

ax.axvline(x=sweet_deg, color=COLORS['true'], linestyle='--',
           alpha=0.8, linewidth=2, label=f'Sweet spot (degree={sweet_deg})')
ax.annotate(f'Sweet Spot\ndeg={sweet_deg}',
            xy=(sweet_deg, sweet_mse), xytext=(sweet_deg + 1.5, sweet_mse + 0.3),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=11)

# Zone annotations
ax.axvspan(1, 3, alpha=0.07, color='blue', label='Underfitting zone (high bias)')
ax.axvspan(8, 15, alpha=0.07, color='red', label='Overfitting zone (high variance)')

ax.text(1.5, ax.get_ylim()[1] * 0.88, 'UNDERFIT\n(High Bias)',
        fontsize=10, color='blue', alpha=0.8, ha='center')
ax.text(11.5, ax.get_ylim()[1] * 0.88, 'OVERFIT\n(High Variance)',
        fontsize=10, color='red', alpha=0.8, ha='center')

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13)
ax.set_ylabel('Mean Squared Error ($100k²)', fontsize=13)
ax.set_title('The Bias-Variance Tradeoff: Training vs. Validation Error\n'
             'House Price Prediction — California Dataset', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(degrees)
plt.tight_layout()
plt.show()

print(f'\nDiagnosis:')
print(f'  Degree 1  → Train MSE={df_results.loc[0,"train_mse"]:.4f}, Val MSE={df_results.loc[0,"val_mse"]:.4f}  → UNDERFIT')
print(f'  Degree {sweet_deg}  → Train MSE={df_results.loc[sweet_idx,"train_mse"]:.4f}, Val MSE={sweet_mse:.4f}  → GOOD FIT')
print(f'  Degree 15 → Train MSE={df_results.iloc[-1]["train_mse"]:.4f}, Val MSE={df_results.iloc[-1]["val_mse"]:.4f}  → OVERFIT')

In [ ]:
# ---------------------------------------------------------------------------
# Section 5: Visual Fit Comparison — Degree 1 vs 5 vs 15
# Show what each model's learned curve looks like against the data.
# ---------------------------------------------------------------------------
x_plot = np.linspace(0, 1, 300).reshape(-1, 1)

showcase_degrees = [1, 5, 15]
titles = ['Degree 1 (Underfitting)\nHigh Bias, Low Variance',
          'Degree 5 (Good Fit)\nBalanced Bias-Variance',
          'Degree 15 (Overfitting)\nLow Bias, High Variance']
colors_fit = [COLORS['val'], COLORS['true'], COLORS['highlight']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, deg, title, col in zip(axes, showcase_degrees, titles, colors_fit):
    # Scatter: train (light) and val (darker)
    ax.scatter(X_train, y_train, alpha=0.5, s=30,
               color=COLORS['train'], label='Train', zorder=3)
    ax.scatter(X_val, y_val, alpha=0.7, s=40,
               color=COLORS['val'], label='Val', zorder=3, marker='D')

    # Fitted curve
    y_plot = fitted_models[deg].predict(x_plot)
    ax.plot(x_plot, y_plot, color=col, linewidth=2.5, label=f'Degree {deg} fit')

    # True underlying function
    y_true_plot = 3.0 + 8.0*x_plot[:,0] - 6.0*x_plot[:,0]**2 + 10.0*x_plot[:,0]**3 + 2.0
    ax.plot(x_plot, y_true_plot, color='black', linewidth=1.5,
            linestyle='--', alpha=0.6, label='True function')

    idx = degrees.index(deg)
    train_err = df_results.loc[idx, 'train_mse']
    val_err   = df_results.loc[idx, 'val_mse']
    ax.set_title(f'{title}\nTrain MSE={train_err:.3f} | Val MSE={val_err:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Normalized Square Footage', fontsize=10)
    ax.set_ylabel('Price ($100k)', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('What Overfitting and Underfitting Look Like on House Price Data',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 6: Bias-Variance Decomposition via Bootstrap
# We estimate Bias^2 and Variance by training the same model on many
# different bootstrap samples of the training data.
# ---------------------------------------------------------------------------
np.random.seed(42)
N_BOOTSTRAP = 50
x_test_pts  = np.linspace(0.1, 0.9, 20).reshape(-1, 1)

# True values at test points
y_true_test = (3.0 + 8.0*x_test_pts[:,0] - 6.0*x_test_pts[:,0]**2
               + 10.0*x_test_pts[:,0]**3 + 2.0)

bias2_list, var_list, total_list = [], [], []

for deg in degrees:
    preds_bootstrap = []   # shape: (N_BOOTSTRAP, n_test_pts)

    for b in range(N_BOOTSTRAP):
        boot_idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
        X_b = X_train[boot_idx]
        y_b = y_train[boot_idx]

        pipe = Pipeline([
            ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
            ('scaler', StandardScaler()),
            ('lr', LinearRegression())
        ])
        pipe.fit(X_b, y_b)
        preds_bootstrap.append(pipe.predict(x_test_pts))

    preds_bootstrap = np.array(preds_bootstrap)  # (N_BOOTSTRAP, n_test_pts)
    mean_pred  = preds_bootstrap.mean(axis=0)     # average prediction per test point

    bias2   = np.mean((mean_pred - y_true_test)**2)
    variance = np.mean(preds_bootstrap.var(axis=0))
    total   = bias2 + variance

    bias2_list.append(bias2)
    var_list.append(variance)
    total_list.append(total)

df_bv = pd.DataFrame({
    'degree': degrees,
    'bias2': bias2_list,
    'variance': var_list,
    'total': total_list
})
print('Bias-Variance Decomposition (bootstrap estimates):')
print(df_bv.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ---------------------------------------------------------------------------
# Section 6b: Bias-Variance Decomposition Plot
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_bv['degree'], df_bv['bias2'], marker='o', linewidth=2.5,
        color='#2196F3', label='Bias² (systematic error)', markersize=8)
ax.plot(df_bv['degree'], df_bv['variance'], marker='s', linewidth=2.5,
        color='#F44336', label='Variance (sensitivity to data)', markersize=8)
ax.plot(df_bv['degree'], df_bv['total'], marker='^', linewidth=2.5,
        color='#4CAF50', linestyle='--', label='Total Error (Bias² + Variance)', markersize=8)

# Shade areas under bias2 and variance
ax.fill_between(df_bv['degree'], 0, df_bv['bias2'], alpha=0.15, color='#2196F3')
ax.fill_between(df_bv['degree'], df_bv['bias2'], df_bv['total'], alpha=0.15, color='#F44336')

# Mark optimal
opt_idx = df_bv['total'].idxmin()
ax.axvline(df_bv.loc[opt_idx, 'degree'], color='black', linestyle=':', alpha=0.7, linewidth=2)
ax.text(df_bv.loc[opt_idx, 'degree'] + 0.3, df_bv['total'].max() * 0.9,
        f'Optimal complexity\n(degree={df_bv.loc[opt_idx,"degree"]})', fontsize=10)

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13)
ax.set_ylabel('Error Component', fontsize=13)
ax.set_title('Bias² vs. Variance vs. Total Error\n'
             'Estimated via Bootstrap on House Price Data', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(degrees)
plt.tight_layout()
plt.show()

print('\nConclusion:')
print('  As degree increases: Bias² falls, Variance rises.')
print(f'  Total error is minimized at degree {df_bv.loc[opt_idx,"degree"]}.')
print('  This is the bias-variance tradeoff in action.')

In [ ]:
# ---------------------------------------------------------------------------
# Section 7: The Wiggly Overfit — Extreme Visualization
# Show 15 data points with degree-15 polynomial: interpolates perfectly but
# wiggles wildly between points (Runge's phenomenon).
# ---------------------------------------------------------------------------
np.random.seed(42)
n_small = 15
x_small = np.sort(np.random.uniform(0, 1, n_small))
y_small = (3.0 + 8.0*x_small - 6.0*x_small**2 + 10.0*x_small**3 + 2.0
           + np.random.normal(0, 0.3, n_small))

x_dense = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, deg, col, label in zip(
    axes,
    [3, 14],
    [COLORS['true'], COLORS['highlight']],
    ['Degree 3 — Generalizes Well', 'Degree 14 — Wildly Overfits']
):
    # Fit
    coeffs = np.polyfit(x_small, y_small, deg)
    y_fit  = np.polyval(coeffs, x_dense)
    y_pts  = np.polyval(coeffs, x_small)

    ax.scatter(x_small, y_small, s=80, zorder=5, color='black',
               label='Training data (n=15)', edgecolors='white', linewidth=1.5)
    ax.plot(x_dense, y_fit, color=col, linewidth=2.5, label=f'{label}')

    # Residual lines
    for xi, yi, yp in zip(x_small, y_small, y_pts):
        ax.plot([xi, xi], [yi, yp], color='gray', alpha=0.5, linewidth=1)

    # True function
    y_true_d = 3.0 + 8.0*x_dense - 6.0*x_dense**2 + 10.0*x_dense**3 + 2.0
    ax.plot(x_dense, y_true_d, 'k--', linewidth=1.5, alpha=0.5, label='True function')

    train_mse = mean_squared_error(y_small, y_pts)
    ax.set_title(f'{label}\nTrain MSE = {train_mse:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Normalized sqft', fontsize=11)
    ax.set_ylabel('Price ($100k)', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_ylim([y_small.min() - 2, y_small.max() + 2])

plt.suptitle('The Overfitting Wiggle: Why Zero Training Error Is Dangerous',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 8: Residual Patterns
# Residuals vs fitted values reveal the quality of fit:
#   - Degree 1:  systematic curve pattern (model is missing nonlinearity)
#   - Degree 5:  random scatter (good fit)
#   - Degree 15: erratic, volatile residuals
# ---------------------------------------------------------------------------
residual_degrees = [1, 5, 15]
titles_res = ['Degree 1 — Underfitting\n(Systematic Residual Pattern)',
              'Degree 5 — Good Fit\n(Random Residuals)',
              'Degree 15 — Overfitting\n(Erratic on Validation Data)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, deg, title in zip(axes, residual_degrees, titles_res):
    y_pred_tr  = fitted_models[deg].predict(X_train)
    y_pred_val = fitted_models[deg].predict(X_val)
    res_train  = y_train  - y_pred_tr
    res_val    = y_val    - y_pred_val

    ax.scatter(y_pred_tr, res_train, alpha=0.6, s=30, color=COLORS['train'],
               label='Train residuals', zorder=3)
    ax.scatter(y_pred_val, res_val, alpha=0.8, s=40, color=COLORS['val'],
               label='Val residuals', zorder=3, marker='D')
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Fitted Values', fontsize=11)
    ax.set_ylabel('Residuals (y − ŷ)', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Residual Plots as Diagnostic Tools', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Reading residual plots:')
print('  Degree 1:  Residuals form a U-shaped curve → model is missing the nonlinear terms')
print('  Degree 5:  Residuals scatter randomly around 0 → good fit, no systematic error')
print('  Degree 15: Val residuals are volatile and spread out → model overfits noise')

In [ ]:
# ---------------------------------------------------------------------------
# Section 9: Overfitting Score = Val Error / Train Error
# A ratio close to 1.0 = good generalization.
# A large ratio = overfitting.
# ---------------------------------------------------------------------------
df_results['overfit_score'] = df_results['val_mse'] / df_results['train_mse']

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(range(len(degrees)), df_results['overfit_score'],
              color=[COLORS['highlight'] if s > 3 else COLORS['true']
                     for s in df_results['overfit_score']],
              edgecolor='white', linewidth=1.5)

# Value labels on bars
for bar, val in zip(bars, df_results['overfit_score']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}x', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(1.0, color='black', linewidth=1.5, linestyle='--', label='Perfect generalization (ratio=1)')
ax.axhline(3.0, color='red', linewidth=1.5, linestyle=':', alpha=0.7, label='Overfitting threshold (ratio=3)')
ax.set_xticks(range(len(degrees)))
ax.set_xticklabels([f'Degree {d}' for d in degrees], fontsize=11)
ax.set_ylabel('Overfitting Score (Val MSE / Train MSE)', fontsize=12)
ax.set_title('Overfitting Score by Model Complexity\n'
             'Purple = Overfitting (ratio > 3x), Green = Healthy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\nOverfitting Score Interpretation:')
for _, row in df_results.iterrows():
    status = 'UNDERFIT' if row['degree'] <= 2 else ('OVERFIT' if row['overfit_score'] > 3 else 'HEALTHY')
    print(f'  Degree {int(row["degree"]):2d}: score={row["overfit_score"]:6.2f}x  → {status}')

In [ ]:
# ---------------------------------------------------------------------------
# Section 10: Sample Size Effect
# Overfitting is worse with less data. Same degree-8 model on n=20 vs n=200.
# ---------------------------------------------------------------------------
np.random.seed(42)
N_large = 200

# Generate large dataset
sqft_large = np.sort(np.random.uniform(0, 1, N_large))
price_large = (3.0 + 8.0*sqft_large - 6.0*sqft_large**2
               + 10.0*sqft_large**3 + 2.0
               + np.random.normal(0, 0.4, N_large))

DEG = 8  # Same model complexity
sample_sizes = [20, 50, 100, 200]
x_dense = np.linspace(0, 1, 300).reshape(-1, 1)

train_errs, val_errs, gaps = [], [], []
best_pipes = {}

for n in sample_sizes:
    idx  = np.random.choice(N_large, size=n, replace=False)
    X_n  = sqft_large[idx].reshape(-1, 1)
    y_n  = price_large[idx]
    Xtr, Xvl, ytr, yvl = train_test_split(X_n, y_n, test_size=0.25, random_state=42)

    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=DEG, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', LinearRegression())
    ])
    pipe.fit(Xtr, ytr)
    tr_err = mean_squared_error(ytr, pipe.predict(Xtr))
    vl_err = mean_squared_error(yvl, pipe.predict(Xvl))
    train_errs.append(tr_err)
    val_errs.append(vl_err)
    gaps.append(vl_err - tr_err)
    best_pipes[n] = (pipe, Xtr, ytr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Error by sample size
ax = axes[0]
ax.plot(sample_sizes, train_errs, marker='o', linewidth=2.5,
        color=COLORS['train'], label='Train MSE')
ax.plot(sample_sizes, val_errs, marker='s', linewidth=2.5,
        color=COLORS['val'], label='Val MSE')
ax.fill_between(sample_sizes, train_errs, val_errs, alpha=0.15, color='red',
                label='Generalization gap')
ax.set_xlabel('Training Set Size (n)', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title(f'Overfitting Worsens with Less Data\n(Fixed Degree-{DEG} Model)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Plot 2: Generalization gap bar chart
ax = axes[1]
bar_colors = [COLORS['highlight'] if g > 0.5 else COLORS['true'] for g in gaps]
bars = ax.bar([str(n) for n in sample_sizes], gaps, color=bar_colors, edgecolor='white')
for bar, val in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Training Set Size (n)', fontsize=12)
ax.set_ylabel('Generalization Gap (Val − Train MSE)', fontsize=12)
ax.set_title('Generalization Gap vs. Sample Size\n(Smaller gap = better generalization)', fontsize=12, fontweight='bold')

plt.suptitle('More Data Reduces Overfitting (Same Model Complexity)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Degree-{DEG} model generalization gap by sample size:')
for n, g in zip(sample_sizes, gaps):
    print(f'  n={n:3d}: gap={g:.4f}  {"← Severe overfitting" if g > 0.5 else "← Manageable"}')

## Section 11: How Regularization Fixes Overfitting

Now that we understand the problem, let's preview the solution. Regularization adds a **penalty term** to the loss function that discourages the model from fitting the training data too closely.

### Regularized Objective

$$\text{Loss} = \underbrace{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{fit the data}} + \underbrace{\lambda \cdot \text{Penalty}(\theta)}_{\text{keep coefficients small}}$$

The penalty term forces the model to balance two competing objectives:
1. **Fit the training data** (minimize residuals)
2. **Keep the model simple** (minimize coefficient magnitudes)

### Two Types of Penalty:

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | $\lambda \sum \theta_j^2$ | Shrinks all coefficients towards zero, none become exactly zero |
| **Lasso (L1)** | $\lambda \sum |\theta_j|$ | Shrinks some coefficients to **exactly zero** (feature selection) |

### Connection to the Bias-Variance Tradeoff
- As $\lambda$ increases: model becomes simpler → **bias increases, variance decreases**
- As $\lambda$ decreases toward 0: model approaches OLS → **bias decreases, variance increases**
- The optimal $\lambda$ minimizes the validation error — just like the optimal polynomial degree

In [ ]:
# ---------------------------------------------------------------------------
# Section 11b: Quick Demo — Ridge Regularization Tames Overfitting
# ---------------------------------------------------------------------------
from sklearn.linear_model import Ridge

np.random.seed(42)
alphas = [0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
x_dense = np.linspace(0, 1, 300).reshape(-1, 1)
DEG_OVER = 12  # Use an overfitting degree

ridge_train, ridge_val = [], []

for alpha in alphas:
    if alpha == 0:
        lr_model = LinearRegression()
    else:
        lr_model = Ridge(alpha=alpha)

    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=DEG_OVER, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', lr_model)
    ])
    pipe.fit(X_train, y_train)
    ridge_train.append(mean_squared_error(y_train, pipe.predict(X_train)))
    ridge_val.append(mean_squared_error(y_val, pipe.predict(X_val)))

fig, ax = plt.subplots(figsize=(10, 5))
alpha_labels = ['OLS\n(α=0)'] + [f'α={a}' for a in alphas[1:]]
x_pos = range(len(alphas))

ax.plot(x_pos, ridge_train, marker='o', linewidth=2.5, color=COLORS['train'],
        label='Train MSE')
ax.plot(x_pos, ridge_val, marker='s', linewidth=2.5, color=COLORS['val'],
        label='Validation MSE')
ax.fill_between(x_pos, ridge_train, ridge_val, alpha=0.15, color='red')

best_alpha_idx = np.argmin(ridge_val)
ax.axvline(best_alpha_idx, color=COLORS['true'], linestyle='--', linewidth=2,
           label=f'Best α = {alphas[best_alpha_idx]}')

ax.set_xticks(x_pos)
ax.set_xticklabels(alpha_labels, fontsize=10)
ax.set_xlabel('Regularization Strength (α)', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title(f'Ridge Regularization Reduces Overfitting (Degree-{DEG_OVER} Model)\n'
             f'As α increases, the generalization gap shrinks', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Best Ridge alpha: {alphas[best_alpha_idx]}')
print(f'Without regularization: Train={ridge_train[0]:.4f}, Val={ridge_val[0]:.4f}, Gap={ridge_val[0]-ridge_train[0]:.4f}')
print(f'With best alpha:        Train={ridge_train[best_alpha_idx]:.4f}, Val={ridge_val[best_alpha_idx]:.4f}, Gap={ridge_val[best_alpha_idx]-ridge_train[best_alpha_idx]:.4f}')

## Section 11c: Cross-Validation — A More Reliable Generalization Estimate

A single train/val split can be noisy — we might get lucky or unlucky with how the split falls. **K-Fold Cross-Validation** gives a more stable estimate by averaging over multiple splits.

### K-Fold CV Algorithm
1. Split the data into K equal folds
2. For each fold k: train on the other K-1 folds, evaluate on fold k
3. Average the K validation scores → more stable estimate of true generalization error

For model selection: pick the degree (or α) with the lowest **mean** CV error.

```
Fold 1:  [VAL   | TRAIN | TRAIN | TRAIN | TRAIN]
Fold 2:  [TRAIN | VAL   | TRAIN | TRAIN | TRAIN]
Fold 3:  [TRAIN | TRAIN | VAL   | TRAIN | TRAIN]
Fold 4:  [TRAIN | TRAIN | TRAIN | VAL   | TRAIN]
Fold 5:  [TRAIN | TRAIN | TRAIN | TRAIN | VAL  ]
```

**Why it matters:** With cross-validation, the model sees every data point as a validation point exactly once, eliminating the variance of a single unlucky split.

In [ ]:
# ---------------------------------------------------------------------------
# Section 11c: K-Fold Cross-Validation for Polynomial Degree Selection
# More reliable than a single train/val split — averages over 5 folds.
# ---------------------------------------------------------------------------
from sklearn.model_selection import cross_val_score, KFold

degrees_cv = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 15]
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

cv_mean_mse = []
cv_std_mse  = []

for deg in degrees_cv:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', LinearRegression())
    ])
    # cross_val_score returns negative MSE (sklearn convention)
    scores = cross_val_score(pipe, X_1d, y, cv=kf,
                             scoring='neg_mean_squared_error')
    mse_scores = -scores  # convert to positive MSE
    cv_mean_mse.append(mse_scores.mean())
    cv_std_mse.append(mse_scores.std())

cv_mean_mse = np.array(cv_mean_mse)
cv_std_mse  = np.array(cv_std_mse)
best_cv_deg = degrees_cv[np.argmin(cv_mean_mse)]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(degrees_cv, cv_mean_mse, marker='o', linewidth=2.5,
        color=COLORS['highlight'], label=f'{K}-Fold CV Mean MSE', markersize=8)
ax.fill_between(degrees_cv,
                cv_mean_mse - cv_std_mse,
                cv_mean_mse + cv_std_mse,
                alpha=0.2, color=COLORS['highlight'], label='±1 std across folds')
ax.axvline(best_cv_deg, color=COLORS['true'], linewidth=2, linestyle='--',
           label=f'Best degree = {best_cv_deg}')
ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('Cross-Validation MSE', fontsize=13)
ax.set_title(f'{K}-Fold Cross-Validation: Selecting Optimal Polynomial Degree\n'
             'Shaded band = variance across folds (narrower = more stable estimate)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(degrees_cv)
plt.tight_layout()
plt.show()

print(f'CV-selected best degree: {best_cv_deg}')
best_idx = degrees_cv.index(best_cv_deg)
print(f'CV MSE at best degree: {cv_mean_mse[best_idx]:.4f} ± {cv_std_mse[best_idx]:.4f}')
print()
print('Comparison across all degrees:')
for d, m, s in zip(degrees_cv, cv_mean_mse, cv_std_mse):
    marker = ' ← BEST' if d == best_cv_deg else ''
    print(f'  Degree {d:2d}: CV MSE = {m:.4f} ± {s:.4f}{marker}')

In [ ]:
# ---------------------------------------------------------------------------
# Section 11d: Coefficient Explosion — Another Signature of Overfitting
# Overfitting models use enormous, cancelling coefficients to fit noise.
# Regularization works precisely because it penalizes large coefficients.
# ---------------------------------------------------------------------------
degrees_coef = [1, 3, 5, 8, 12, 15]
max_coef_list = []
coef_data     = []

for deg in degrees_coef:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr', LinearRegression())
    ])
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps['lr'].coef_
    max_coef_list.append(np.max(np.abs(coefs)))
    coef_data.append(np.abs(coefs))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Max |coefficient| vs degree (log scale)
ax = axes[0]
ax.semilogy(degrees_coef, max_coef_list, marker='o', linewidth=2.5,
            color=COLORS['highlight'], markersize=10)
for deg, val in zip(degrees_coef, max_coef_list):
    ax.text(deg + 0.2, val * 1.6, f'{val:.1f}', ha='left', fontsize=10, fontweight='bold')
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('Max |Coefficient| (log scale)', fontsize=12)
ax.set_title('Coefficient Magnitudes Explode with Degree\n'
             'Log scale — notice exponential growth', fontsize=12, fontweight='bold')
ax.set_xticks(degrees_coef)

# Plot 2: Boxplots of coefficient distributions
ax = axes[1]
bp = ax.boxplot(coef_data, labels=[str(d) for d in degrees_coef],
                patch_artist=True,
                boxprops=dict(facecolor=COLORS['highlight'], alpha=0.55),
                medianprops=dict(color='black', linewidth=2),
                flierprops=dict(marker='o', markersize=4, alpha=0.5))
ax.set_yscale('log')
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('|Coefficient| (log scale)', fontsize=12)
ax.set_title('Distribution of |Coefficients| by Degree\n'
             'Overfitting → heavy upper tail of extreme weights', fontsize=12, fontweight='bold')

plt.suptitle('Coefficient Explosion: A Third Diagnostic for Overfitting\n'
             'Regularization cures overfitting by penalizing large coefficients',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Summary: Coefficient magnitudes by degree')
for deg, mc in zip(degrees_coef, max_coef_list):
    bar = '█' * min(int(mc / max_coef_list[-1] * 30), 30)
    print(f'  Degree {deg:2d}: max|coef| = {mc:8.2f}  {bar}')
print(f'\n  Ratio degree-15 / degree-1: {max_coef_list[-1]/max_coef_list[0]:.0f}x larger!')
print('  This is why Ridge/Lasso (which penalize large coefficients) reduce overfitting.')

## Section 12: Summary & Key Takeaways

### What We Learned

| Concept | Definition | Diagnosis | Fix |
|---------|-----------|-----------|-----|
| **Underfitting** | Model too simple | High train AND val error | Increase complexity, add features |
| **Overfitting** | Model too complex | Low train, high val error | Regularize, get more data, reduce complexity |
| **Good fit** | Balanced complexity | Low train AND val error, small gap | — |

### The Bias-Variance Tradeoff
$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \sigma^2_{\text{irreducible}}$$
- **Simpler models** → High Bias, Low Variance (underfitting)
- **Complex models** → Low Bias, High Variance (overfitting)
- **Regularization** moves the optimal point by increasing the effective model constraint

### Rules of Thumb
1. **Always** hold out a validation set before comparing models
2. If `val_err >> train_err` → regularize or collect more data
3. If both errors are large → increase model complexity or engineer better features
4. More training data almost always helps overfitting (shifts the sweet spot right)
5. Regularization is free: always worth trying before adding data

## Self-Check Questions

Test your understanding. Try to answer each question before expanding the solution.

---

**Question 1:**  
A decision tree achieves 100% accuracy on training data. Is it definitely overfitting? What's the **only** way to know for sure?

<details>
<summary>Click to reveal answer</summary>

**Not necessarily — but it is highly suspicious.** A decision tree with no depth limit will always achieve 100% training accuracy by creating a leaf for each training point. This is *almost certainly* overfitting.

**The only way to know for sure is to evaluate it on held-out test data.**

If test accuracy ≈ 100% too → it genuinely learned the patterns (or the task is very easy).  
If test accuracy << 100% → it memorized training noise (overfitting confirmed).

Key principle: **Training error is not a reliable measure of generalization.** You must always measure performance on data the model has never seen. This is why train/val/test splits and cross-validation exist.

```python
# Always do this:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
dt = DecisionTreeClassifier()  # No max_depth → will overfit
dt.fit(X_train, y_train)
print(dt.score(X_train, y_train))  # Will be 1.0
print(dt.score(X_test, y_test))    # THIS is what matters
```
</details>

---

**Question 2:**  
Model A has Bias²=0.05, Variance=0.40. Model B has Bias²=0.35, Variance=0.08. Both have total error ≈ 0.45. Which would you prefer in production and why?

<details>
<summary>Click to reveal answer</summary>

**It depends on context, but in most production scenarios, Model B (high bias, low variance) is preferable.**

**Argument for Model B (low variance):**
- **Stability:** Model B's predictions don't change much if you retrain on slightly different data. This is crucial in production where data drifts over time.
- **Reproducibility:** If you retrain weekly, Model B gives consistent results. Model A might give wildly different predictions each week.
- **Debugging:** High variance models are harder to debug — the same input might give different predictions across model versions.
- **Regulatory risk:** In finance, healthcare, or legal settings, unstable predictions create legal exposure.

**When Model A (low bias) might be better:**
- If you have huge amounts of fresh data that gets retrained often (the variance averages out over many samples)
- When systematic errors (bias) are more costly than variance (e.g., a systematic underestimate of drug dosage is worse than variable estimates)
- When you're ensembling many models — ensemble methods average out variance, so high-variance models can be excellent base learners

**Key formula reminder:** Total error = Bias² + Variance + σ². Both models have the same total error *in expectation*, but they fail in different ways.
</details>

---

**Question 3:**  
Adding 3× more training data to an overfitting model typically does what to: (a) training error, (b) validation error?

<details>
<summary>Click to reveal answer</summary>

**(a) Training error INCREASES (gets worse)**  
**(b) Validation error DECREASES (gets better)**

**Why training error increases:**  
When the model was overfitting on a small dataset, it could memorize every point (≈ zero training error). With 3× more data, there are too many points to memorize — the model is forced to find generalizable patterns instead of fitting noise. Training error rises toward the true irreducible noise level.

**Why validation error decreases:**  
More training data means the model sees more diverse examples of the underlying distribution. The model learns the true signal better and becomes less sensitive to the particular noise in the training sample. Variance decreases, so generalization improves.

**Visual intuition from our plots:**  
In Section 10, we saw this empirically: as n increased from 20 to 200 with the same degree-8 model, the generalization gap (val_err − train_err) shrank dramatically. The validation error improved while training error increased slightly.

**The two curves converge:** As n → ∞, both training and validation error converge to the irreducible noise floor σ². You cannot beat irreducible noise no matter how much data you have.
</details>